# Part 2: Data Analysis

Analysis of the Buienradar weather data stored in `buienradar.db`.

In [6]:
import sqlite3

DB_NAME = "buienradar.db"


def query_db(query: str):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute(query)
        return cursor.fetchall()

## Question 5: Which weather station recorded the highest temperature?

In [7]:
result_5 = query_db("""
    WITH max_temp AS (
        SELECT MAX(temperature) AS max_temperature
        FROM MEASUREMENTS
    )
    SELECT s.stationname, mt.max_temperature
    FROM max_temp mt
    JOIN MEASUREMENTS m ON m.temperature = mt.max_temperature
    JOIN STATIONS s ON s.stationid = m.stationid
""")

for station, temp in result_5:
    print(f"  {station} | {temp} \u00b0C")

  Meetstation Woensdrecht | 17.4 °C


## Question 6: What is the average temperature?

In [8]:
result_6 = query_db("""
    SELECT AVG(temperature) AS avg_temperature
    FROM MEASUREMENTS
""")

print(f"  {result_6[0][0]:.2f} \u00b0C (across all stations and timestamps)")

  14.13 °C (across all stations and timestamps)


### Average temperature per station

In [9]:
result_6b = query_db("""
    WITH avg_temp_per_station AS (
        SELECT stationid, AVG(temperature) AS avg_temperature
        FROM MEASUREMENTS
        GROUP BY stationid
    )
    SELECT s.stationname, atps.avg_temperature
    FROM avg_temp_per_station atps
    JOIN STATIONS s ON s.stationid = atps.stationid
    ORDER BY atps.avg_temperature DESC
""")

for station, avg_temp in result_6b:
    if avg_temp is not None:
        print(f"  {station:<30} {avg_temp:.2f} \u00b0C")

  Meetstation Woensdrecht        17.40 °C
  Meetstation Ell                17.20 °C
  Meetstation Westdorpe          16.90 °C
  Meetstation Arnhem             16.60 °C
  Meetstation Eindhoven          16.60 °C
  Meetstation Volkel             16.50 °C
  Meetstation Gilze Rijen        16.30 °C
  Meetstation Maastricht         16.30 °C
  Meetstation Herwijnen          15.80 °C
  Meetstation De Bilt            15.70 °C
  Meetstation Rotterdam          15.70 °C
  Meetstation Lopik-Cabauw       15.70 °C
  Meetstation Horst              15.60 °C
  Meetstation Groenlo-Hupsel     15.40 °C
  Meetstation Twente             15.40 °C
  Meetstation Hoogeveen          15.00 °C
  Meetstation Heino              14.90 °C
  Meetstation Vlissingen         14.80 °C
  Meetstation Goes               14.80 °C
  Meetstation Marknesse          14.30 °C
  Meetstation Lelystad           13.90 °C
  Meetstation Schiphol           13.70 °C
  Meetstation Groningen          13.70 °C
  Meetstation Voorschoten        1

## Question 7: Which station has the biggest difference between feel temperature and actual temperature?

In [10]:
result_7 = query_db("""
    SELECT s.stationname, MAX(ABS(feeltemperature - temperature)) AS max_diff
    FROM MEASUREMENTS m
    JOIN STATIONS s ON s.stationid = m.stationid
    GROUP BY s.stationname
    ORDER BY max_diff DESC
    LIMIT 1
""")

for station, diff in result_7:
    if diff is not None:
        print(f"  {station} | \u0394 {diff:.2f} \u00b0C")

  Meetstation Hoek van Holland | Δ 2.70 °C


## Question 8: Which weather station is located in the North Sea?

In [11]:
result_8 = query_db("""
    SELECT stationname
    FROM STATIONS
    WHERE regio = 'Noordzee'
""")

for (station,) in result_8:
    print(f"  {station}")

  Meetstation Zeeplatform F-3
